## **PROJET TINDER - EDA**

### **5. VISUALISATIONS FINALES & STORYTELLING**

- Input  : `outputs/data/df_speed_dating_cleaned.csv` 
- Livrable : visualisations prêtes pour une présentation équipe marketing

In [22]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path
 
pio.templates.default = "plotly_white"
 
# ── Palette ──────────────────────────────────────────────────────────────────
C = {
    "match":    "#1D9E75",
    "no_match": "#E24B4A",
    "male":     "#378ADD",
    "female":   "#D4537E",
    "purple":   "#7F77DD",
    "amber":    "#EF9F27",
    "teal":     "#0FA98E",
    "coral":    "#D85A30",
    "neutral":  "#888780",
    "dark":     "#1A1A2E",
    "light_bg": "#F8F7F4",
}
 
ATTR6    = ["attr",  "sinc",  "intel",  "fun",   "amb",   "shar"]
ATTR6_FR = ["Attractivité","Sincérité","Intelligence","Fun","Ambition","Intérêts communs"]
PREF_COLS = ["attr1_1","sinc1_1","intel1_1","fun1_1","amb1_1","shar1_1"]
SCORE_OTHER = ["attr_o","sinc_o","intel_o","fun_o","amb_o","shar_o","like_o","prob_o"]
SCORE_SELF  = ["attr","sinc","intel","fun","amb","shar","like","prob"]

PROJECT_ROOT = Path('../')
INPUT_DATA_PATH = PROJECT_ROOT / 'outputs' / 'data' / 'df_speed_dating_prep.csv'
print(INPUT_DATA_PATH)

../outputs/data/df_speed_dating_prep.csv


In [23]:
# =============================================================================
# CHARGEMENT + PREP
# =============================================================================
df = pd.read_csv(INPUT_DATA_PATH)
print(f"\nDataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
 
df["age_diff"] = (df["age"] - df["age_o"]).abs()
df["score_received_avg"] = df[SCORE_OTHER].mean(axis=1).round(2)
df["attr_expectation_gap"] = (df["attr1_1"] - df["attr_o"]).round(2)
df["gender_label"] = df["gender"].map({0:"Femme", 1:"Homme"})
df["match_label"] = df["match"].map({0:"Pas de match", 1:"Match"})
 
# ── Prétraitements ───────
match_rate = df["match"].mean() * 100
dec_f = df[df["gender"]==0]["dec"].mean() * 100
dec_m = df[df["gender"]==1]["dec"].mean() * 100
 
deltas, corrs = {}, {}
for col in ATTR6:
    m0 = df[df["match"]==0][col].mean()
    m1 = df[df["match"]==1][col].mean()
    deltas[col] = round(m1 - m0, 3)
    corrs[col]  = round(df[col].corr(df["match"]), 3)
 
like_thresholds = [4, 5, 6, 7, 8, 9]
like_rates = [df[df["like"]>=t]["match"].mean()*100 for t in like_thresholds]
like_ratio = df[df["like"]>=8]["match"].mean() / df["match"].mean()
 
gender_deltas = {}
for col in ATTR6[:5]:
    df_f = df[df["gender"]==0]
    df_m = df[df["gender"]==1]
    gender_deltas[col] = (
        df_f[df_f["match"]==1][col].mean() - df_f[df_f["match"]==0][col].mean(),
        df_m[df_m["match"]==1][col].mean() - df_m[df_m["match"]==0][col].mean(),
    )
 
# Pref T1 vs réel
decl_f, reel_f, decl_m, reel_m = [], [], [], []
for p, s in zip(PREF_COLS[:5], ATTR6[:5]):
    decl_f.append(df[df["gender"]==0][p].mean())
    reel_f.append(df[df["gender"]==0][s].mean())
    decl_m.append(df[df["gender"]==1][p].mean())
    reel_m.append(df[df["gender"]==1][s].mean())
 
print("Prétraitements terminés")


Dataset : 8,378 lignes × 48 colonnes
Prétraitements terminés


In [24]:
# =============================================================================
# DASHBOARD EXÉCUTIF
# =============================================================================
print("\n" + "─" * 65)
print("DASHBOARD EXÉCUTIF")
print("─" * 65)
 
 
best_attr   = max(deltas, key=deltas.get)
best_delta  = deltas[best_attr]
max_t1_corr = max(abs(df[p].corr(df["match"])) for p in PREF_COLS)
like8_rate  = df[df["like"]>=8]["match"].mean() * 100
 
# 4 KPI indicators
fig_v1 = go.Figure()
kpis = [
    {"mode":"gauge+number","value":match_rate,
     "title":{"text":"Taux match global","font":{"size":13}},
     "number":{"suffix":"%","font":{"size":32,"color":C["match"]}},
     "gauge":{"axis":{"range":[0,50]},"bar":{"color":C["match"]}},
     "domain":{"x":[0.0,0.24],"y":[0.0,1.0]}},
    {"mode":"number","value":best_delta,
     "title":{"text":f"Différence {ATTR6_FR[ATTR6.index(best_attr)]}<br>(meilleur attribut)","font":{"size":13}},
     "number":{"prefix":"+","valueformat":".2f","font":{"size":36,"color":C["amber"]}},
     "domain":{"x":[0.26,0.50],"y":[0.0,1.0]}},
    {"mode":"number","value":max_t1_corr,
     "title":{"text":"Max corr. préf. T1 vs match<br>(déclarer ≠ prédire)","font":{"size":13}},
     "number":{"suffix":"~ 0","valueformat":".3f","font":{"size":36,"color":C["no_match"]}},
     "domain":{"x":[0.52,0.74],"y":[0.0,1.0]}},
    {"mode":"number","value":like8_rate,
     "title":{"text":"Match rate si like≥8<br>(2.1x la base)","font":{"size":13}},
     "number":{"suffix":"%","valueformat":".1f","font":{"size":36,"color":C["purple"]}},
     "domain":{"x":[0.76,1.0],"y":[0.0,1.0]}},
]
for kpi in kpis:
    fig_v1.add_trace(go.Indicator(**kpi))
fig_v1.update_layout(
    title={"text":"Dashboard exécutif : 4 KPIs clés",
           "font":{"size":18,"color":C["dark"]},"x":0.5,"xanchor":"center"},
    height=280, paper_bgcolor=C["light_bg"],
)
fig_v1.show()
 
# Barre delta attributs
attrs_sorted = sorted(ATTR6, key=lambda c: deltas[c], reverse=True)
delta_colors_v1 = [C["match"] if deltas[c]>=1.3 else C["amber"] if deltas[c]>=0.9
                   else C["neutral"] for c in attrs_sorted]

fig_v1b = go.Figure(go.Bar(
    x=[ATTR6_FR[ATTR6.index(c)] for c in attrs_sorted],
    y=[deltas[c] for c in attrs_sorted],
    marker_color=delta_colors_v1,
    text=[f"+{deltas[c]:.2f}" for c in attrs_sorted],
    textposition="outside", cliponaxis=False,
))
fig_v1b.add_hline(y=1.0, line_dash="dot", line_color=C["match"],
    annotation_text="Signal fort")
fig_v1b.update_layout(
    title={"text":"Force prédictive des attributs (match=1 vs match=0)",
           "font":{"size":15,"color":C["dark"]},"x":0.5,"xanchor":"center"},
    yaxis_title="Delta score (0-10)", height=360,
    plot_bgcolor=C["light_bg"], paper_bgcolor=C["light_bg"], showlegend=False,
)
fig_v1b.show()


─────────────────────────────────────────────────────────────────
DASHBOARD EXÉCUTIF
─────────────────────────────────────────────────────────────────


In [25]:
# =============================================================================
# PODIUM DES ATTRIBUTS
# =============================================================================
print("\n" + "─" * 65)
print("PODIUM DES ATTRIBUTS")
print("─" * 65)
 
# Tri par delta décroissant
sorted_pairs = sorted(zip(ATTR6_FR, ATTR6), key=lambda x: deltas[x[1]], reverse=True)
labels_sorted  = [fr for fr, _ in sorted_pairs]
deltas_sorted  = [deltas[col] for _, col in sorted_pairs]
corrs_sorted   = [corrs[col]  for _, col in sorted_pairs]
 
bar_colors = []
for d in deltas_sorted:
    if d >= 1.3:   bar_colors.append(C["match"])
    elif d >= 0.9: bar_colors.append(C["amber"])
    else:          bar_colors.append(C["neutral"])
 
fig_v2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Match rate moyen",
        "Corrélation avec match",
    ],
    horizontal_spacing=0.12,
)
 
# Delta bars
fig_v2.add_trace(go.Bar(
    y=labels_sorted[::-1],
    x=deltas_sorted[::-1],
    orientation="h",
    marker_color=bar_colors[::-1],
    marker_line_width=0,
    text=[f"+{d:.3f}" for d in deltas_sorted[::-1]],
    textposition="outside", cliponaxis=False,
    textfont=dict(size=13, color=C["dark"]),
    showlegend=False,
), row=1, col=1)
fig_v2.add_vline(x=1.0, line_dash="dot", line_color=C["match"],
    annotation_text="Seuil fort", row=1, col=1)
fig_v2.add_vline(x=0.7, line_dash="dot", line_color=C["amber"],
    annotation_text="Seuil modéré", row=1, col=1)
 
# Corrélation dots
fig_v2.add_trace(go.Scatter(
    y=labels_sorted[::-1],
    x=corrs_sorted[::-1],
    mode="markers+text",
    marker=dict(
        color=bar_colors[::-1],
        size=[40 if d >= 1.3 else 28 if d >= 0.9 else 20
              for d in deltas_sorted[::-1]],
        symbol="circle",
        line=dict(width=2, color="white"),
    ),
    text=[f"{c:+.3f}" for c in corrs_sorted[::-1]],
    textposition="middle center",
    textfont=dict(size=11, color="white", family="monospace"),
    showlegend=False,
), row=1, col=2)

fig_v2.add_vline(x=0.2, line_dash="dot", line_color=C["match"],
    annotation_text="Signal fort", row=1, col=2)
 
fig_v2.update_layout(
    title={
        "text": "Classement des attributs : Fun & Attractivité dominent, "
                "Sincérité & Intelligence sont des signaux faibles",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=420, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
)
fig_v2.update_xaxes(title_text="score (0-10)", row=1, col=1)
fig_v2.update_xaxes(title_text="Corrélation de Pearson", row=1, col=2)
fig_v2.show()


─────────────────────────────────────────────────────────────────
PODIUM DES ATTRIBUTS
─────────────────────────────────────────────────────────────────


In [26]:
# =============================================================================
# CE QU'ON DIT VS CE QU'ON FAIT
# =============================================================================
print("\n" + "─" * 65)
print("CE QU'ON DIT VS CE QU'ON FAIT")
print("─" * 65)
 
attrs5_fr = ["Attractivité","Sincérité","Intelligence","Fun","Ambition"]
 
fig_v3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Femmes — déclaré T1 vs notes réelles soirée",
        "Hommes — déclaré T1 vs notes réelles soirée",
    ],
    horizontal_spacing=0.14,
)
 
for col_idx, (gval, d_vals, r_vals, clr) in enumerate([
    (0, decl_f, reel_f, C["female"]),
    (1, decl_m, reel_m, C["male"]),
]):
    fig_v3.add_trace(go.Bar(
        x=attrs5_fr, y=d_vals,
        name="Déclaré avant (T1)",
        marker_color=clr,
        marker_pattern_shape="/",
        opacity=0.55,
        showlegend=(col_idx == 0),
        legendgroup="declared",
    ), row=1, col=col_idx+1)
    fig_v3.add_trace(go.Bar(
        x=attrs5_fr, y=r_vals,
        name="Noté réellement (soirée)",
        marker_color=clr,
        opacity=0.95,
        showlegend=(col_idx == 0),
        legendgroup="real",
    ), row=1, col=col_idx+1)
 
fig_v3.update_layout(
    title={
        "text": "Ce qu'on dit vouloir vs ce qu'on fait\n",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=440, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
    barmode="group", legend_title="Type de mesure",
)

fig_v3.update_yaxes(title_text="Score moyen (0-10)")
fig_v3.add_annotation(
    text="Les scores 'réels' sont bien plus hauts car les gens notent généreusement "
         "en face à face — mais c'est la VARIANCE qui prédit le match, pas le niveau absolu.",
    xref="paper", yref="paper", x=0.5, y=-0.18,
    showarrow=False, font=dict(size=11, color=C["neutral"]),
    align="center",
)
fig_v3.show()


─────────────────────────────────────────────────────────────────
CE QU'ON DIT VS CE QU'ON FAIT
─────────────────────────────────────────────────────────────────


In [27]:
# =============================================================================
# LE SUPERPRÉDICTEUR LIKE
# =============================================================================
print("\n" + "─" * 65)
print("LE SUPERPRÉDICTEUR LIKE")
print("─" * 65)
 
fig_v4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Taux de match selon le score 'like' minimum",
        "Distribution du score like par résultat",
    ],
    horizontal_spacing=0.12,
)
 
# Courbe taux match
fig_v4.add_trace(go.Scatter(
    x=like_thresholds, y=like_rates,
    mode="lines+markers",
    name="Taux de match",
    marker=dict(color=C["match"], size=12,
                line=dict(width=2, color="white")),
    line=dict(color=C["match"], width=3),
    fill="tozeroy", fillcolor="rgba(29,158,117,0.12)",
    showlegend=False,
), row=1, col=1)
 
fig_v4.add_hline(
    y=match_rate, line_dash="dash", line_color=C["neutral"],
    line_width=2,
    annotation_text=f"Base : {match_rate:.1f}%",
    annotation_position="bottom right",
    row=1, col=1,
)
 
# Zone d'action Tinder
fig_v4.add_vrect(
    x0=7.5, x1=9.5,
    fillcolor="rgba(29,158,117,0.1)",
    line_width=0,
    annotation_text="Zone Tinder",
    annotation_position="top left",
    row=1, col=1,
)
 
# Annotations clés
for t, r in zip(like_thresholds, like_rates):
    fig_v4.add_annotation(
        x=t, y=r + 1.2,
        text=f"{r:.1f}%",
        showarrow=False,
        font=dict(size=11, color=C["dark"], family="monospace"),
        row=1, col=1,
    )
 
# Violin like par match
for mv, clr, nm in [(0, C["no_match"], "Pas de match"), (1, C["match"], "Match")]:
    fig_v4.add_trace(go.Box(
        y=df[df["match"]==mv]["like"],
        name=nm,
        marker_color=clr,
        boxmean=True, 
        opacity=0.82, width=0.7,
    ), row=1, col=2)
 
fig_v4.update_layout(
    title={
        "text": f"like ≥ 8 => {like8_rate:.0f}% de chance de match "
                f"({like_ratio:.1f}x la moyenne) — Le superprédicteur du dataset",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=440, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
    legend_title="Résultat",
)
fig_v4.update_xaxes(title_text="Seuil minimum de like", row=1, col=1,
    tickmode="array", tickvals=like_thresholds)
fig_v4.update_yaxes(title_text="Taux de match (%)", row=1, col=1)
fig_v4.update_yaxes(title_text="Score like (0-10)", row=1, col=2)
fig_v4.show()


─────────────────────────────────────────────────────────────────
LE SUPERPRÉDICTEUR LIKE
─────────────────────────────────────────────────────────────────


In [28]:
# =============================================================================
# RADAR HOMME VS FEMME
# =============================================================================
print("\n" + "─" * 65)
print("RADAR HOMME VS FEMME")
print("─" * 65)
 
attrs_radar = ATTR6_FR[:5]
delta_f_vals = [gender_deltas[c][0] for c in ATTR6[:5]]
delta_m_vals = [gender_deltas[c][1] for c in ATTR6[:5]]
 
fig_v5 = make_subplots(
    rows=1, cols=2,
    specs=[[{"type":"polar"}, {"type":"polar"}]],
    subplot_titles=[
        "Femme",
        "Homme",
    ],
)
 
for col_idx, (vals, clr, genre) in enumerate([
    (delta_f_vals, C["female"], "Femme"),
    (delta_m_vals, C["male"],   "Homme"),
]):
    fig_v5.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=attrs_radar + [attrs_radar[0]],
        fill="toself",
        name=genre,
        line_color=clr,
        fillcolor="rgba(55,138,221,0.20)" if clr == C["male"] else "rgba(212,83,126,0.20)",
        marker=dict(size=8, color=clr),
    ), row=1, col=col_idx+1)
 
fig_v5.update_layout(
    title={
        "text": "Sensibilité: Femmes plus sensibles au Fun & Attractivité",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=480,
    paper_bgcolor=C["light_bg"],
    polar=dict(radialaxis=dict(range=[0, 2], tickfont=dict(size=9))),
    polar2=dict(radialaxis=dict(range=[0, 2], tickfont=dict(size=9))),
    showlegend=False,
)
fig_v5.show()


─────────────────────────────────────────────────────────────────
RADAR HOMME VS FEMME
─────────────────────────────────────────────────────────────────


In [29]:
# =============================================================================
# MYTHES VS RÉALITÉ (ce qui ne prédit PAS le match)
# =============================================================================
print("\n" + "─" * 65)
print("MYTHES VS RÉALITÉ")
print("─" * 65)
 
mythe_vars  = ["Même race", "Intérêts communs", "Écart d'âge", "Ce qu'on déclare vouloir"]
mythe_corrs = [
    df["samerace"].corr(df["match"]),
    df["int_corr"].corr(df["match"]),
    -df["age_diff"].corr(df["match"]),
    max(abs(df[p].corr(df["match"])) for p in PREF_COLS),
]
reel_vars  = ["Fun", "Attractivité", "Intérêts communs (réel)", "Appréciation globale (like)"]
reel_corrs = [corrs["fun"], corrs["attr"], corrs["shar"],
              df["like"].corr(df["match"])]
 
fig_v6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Ce qui ne prédit pas le match (corrélation ~ 0)",
        "Ce qui prédit vraiment le match",
    ],
    horizontal_spacing=0.14,
)
 
fig_v6.add_trace(go.Bar(
    x=mythe_vars, y=[abs(c) for c in mythe_corrs],
    marker_color=C["no_match"], opacity=0.75,
    text=[f"{abs(c):.3f}" for c in mythe_corrs],
    textposition="outside", cliponaxis=False, showlegend=False,
), row=1, col=1)
 
fig_v6.add_trace(go.Bar(
    x=reel_vars, y=reel_corrs,
    marker_color=C["match"], opacity=0.85,
    text=[f"+{c:.3f}" for c in reel_corrs],
    textposition="outside", cliponaxis=False, showlegend=False,
), row=1, col=2)
 
fig_v6.add_hline(y=0.05, line_dash="dot", line_color=C["neutral"],
    annotation_text="Seuil négligeable", row=1, col=1)
 
fig_v6.update_layout(
    title={
        "text": "Mythes vs Réalité : la race, les intérêts communs et "
                "les préférences déclarées ne prédisent pas le match",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=420, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
)
fig_v6.update_yaxes(title_text="Corrélation avec match (valeur absolue)")
fig_v6.show()


─────────────────────────────────────────────────────────────────
MYTHES VS RÉALITÉ
─────────────────────────────────────────────────────────────────


In [30]:
# =============================================================================
# SYNTHÈSE CONSOLE — STORYTELLING FINAL
# =============================================================================
print(f"""
{'=' * 65}
STORYTELLING FINAL TERMINÉ
{'=' * 65}

Sur 8 378 interactions de speed dating à Columbia, seulement 16.5% débouchent sur un match mutuel.
Ce qui crée un match ? Pas ce que les gens croient.
 
- La race en commun : +1.7pp seulement. Négligeable.
- Les intérêts communs : corr=0.031. Quasi nul.
- Ce qu'on déclare vouloir : corr max={max(abs(df[p].corr(df['match'])) for p in PREF_COLS):.3f} ≈ 0.

Les gens ne savent pas ce qui les attire.
 
- Ce qui marche vraiment :
    - Fun          ->  Delta=+1.41 si match
    - Attractivité  ->  Delta=+1.34 si match
    - Intérêts réels  ->  Delta=+1.37 (noté EN FACE, pas déclaré)
    - Like ≥ 8     ->  {like8_rate:.0f}% de match ({like_ratio:.1f}x la base)
 
 Recommandation #1 : Pondérer Fun & Attractivité dans l'algo.
 Recommandation #2 : Créer un 'like score' composite.
 Recommandation #3 : Simplifier l'onboarding — la race et les intérêts déclarés n'ont pas de valeur prédictive."
""")


STORYTELLING FINAL TERMINÉ

Sur 8 378 interactions de speed dating à Columbia, seulement 16.5% débouchent sur un match mutuel.
Ce qui crée un match ? Pas ce que les gens croient.

- La race en commun : +1.7pp seulement. Négligeable.
- Les intérêts communs : corr=0.031. Quasi nul.
- Ce qu'on déclare vouloir : corr max=0.015 ≈ 0.

Les gens ne savent pas ce qui les attire.

- Ce qui marche vraiment :
    - Fun          ->  Delta=+1.41 si match
    - Attractivité  ->  Delta=+1.34 si match
    - Intérêts réels  ->  Delta=+1.37 (noté EN FACE, pas déclaré)
    - Like ≥ 8     ->  35% de match (2.1x la base)

 Recommandation #1 : Pondérer Fun & Attractivité dans l'algo.
 Recommandation #2 : Créer un 'like score' composite.
 Recommandation #3 : Simplifier l'onboarding — la race et les intérêts déclarés n'ont pas de valeur prédictive."



## Synthèse marketing lisible (à partir du CSV clean)

Cette section propose une lecture **non technique** des résultats issus de `outputs/data/df_speed_dating_cleaned.csv`.

Objectifs :
- rendre les graphes plus simples à interpréter,
- distinguer clairement analyses **univariées** et **bivariées**,
- fournir des **observations actionnables** pour les équipes marketing,
- produire un **dashboard final** exporté dans `outputs/images`.

In [31]:
# ============================================================================
# SETUP SECTION MARKETING
# ============================================================================
from pathlib import Path
import shutil
import subprocess
import tempfile
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path("../")
INPUT_CLEAN_PATH = PROJECT_ROOT / "outputs" / "data" / "df_speed_dating_cleaned.csv"
OUTPUT_IMG_DIR = PROJECT_ROOT / "outputs" / "images"
OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

df_mkt = pd.read_csv(INPUT_CLEAN_PATH)

# Labels lisibles
if "gender_label" not in df_mkt.columns and "gender" in df_mkt.columns:
    df_mkt["gender_label"] = df_mkt["gender"].map({0: "Femme", 1: "Homme"})
if "match_label" not in df_mkt.columns and "match" in df_mkt.columns:
    df_mkt["match_label"] = df_mkt["match"].map({0: "Pas de match", 1: "Match"})

# Segmentation demandée : Homme->Femme et Femme->Homme
if "gender" in df_mkt.columns:
    df_mkt["segment_genre"] = df_mkt["gender"].map({1: "Homme -> Femme", 0: "Femme -> Homme"})

ATTR_MAP = {
    "attr": "Attractivite",
    "sinc": "Sincerite",
    "intel": "Intelligence",
    "fun": "Fun",
    "amb": "Ambition",
    "shar": "Interets communs",
}
DECLARED_MAP = {
    "attr1_1": "Attractivite",
    "sinc1_1": "Sincerite",
    "intel1_1": "Intelligence",
    "fun1_1": "Fun",
    "amb1_1": "Ambition",
    "shar1_1": "Interets communs",
}

# Différences entre partenaires
if {"attr", "attr_o"}.issubset(df_mkt.columns):
    df_mkt["gap_attr"] = (df_mkt["attr"] - df_mkt["attr_o"]).abs()
if {"like", "like_o"}.issubset(df_mkt.columns):
    df_mkt["gap_like"] = (df_mkt["like"] - df_mkt["like_o"]).abs()


def export_figure(fig, name_no_ext: str):
    """Export PNG: priorité Kaleido, fallback Chromium headless."""
    png_path = OUTPUT_IMG_DIR / f"{name_no_ext}.png"

    # 1) Export natif Plotly/Kaleido
    try:
        fig.write_image(str(png_path), scale=2)
        print(f"Export image (kaleido): {png_path}")
        return
    except Exception as kaleido_err:
        print(f"Kaleido indisponible ({type(kaleido_err).__name__}), fallback Chromium...")

    # 2) Fallback Chromium headless (toujours PNG)
    chromium_bin = shutil.which("chromium") or shutil.which("chromium-browser") or shutil.which("google-chrome")
    if not chromium_bin:
        raise RuntimeError(
            "Export PNG impossible: ni Kaleido fonctionnel ni Chromium détecté."
        ) from kaleido_err

    with tempfile.NamedTemporaryFile(suffix=".html", delete=False) as tmp:
        tmp_html = Path(tmp.name)

    try:
        fig.write_html(str(tmp_html), include_plotlyjs="cdn")
        cmd = [
            chromium_bin,
            "--headless",
            "--disable-gpu",
            "--no-sandbox",
            "--window-size=1800,1200",
            f"--screenshot={png_path}",
            tmp_html.resolve().as_uri(),
        ]
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"Export image (chromium): {png_path}")
    except Exception as chrome_err:
        raise RuntimeError(
            "Export PNG impossible: echec Kaleido et Chromium headless."
        ) from chrome_err
    finally:
        try:
            tmp_html.unlink(missing_ok=True)
        except Exception:
            pass

print(f"Dataset marketing: {df_mkt.shape[0]:,} lignes x {df_mkt.shape[1]} colonnes")
print(f"Source: {INPUT_CLEAN_PATH}")

Dataset marketing: 541 lignes x 50 colonnes
Source: ../outputs/data/df_speed_dating_cleaned.csv


## 1) Analyses univariées

Ces graphes décrivent la structure générale des données (sans croisement complexe) pour donner un contexte business rapide.

In [32]:
# ============================================================================
# UNIVARIÉ : Match rate global + Importance déclarée vs genre + Distributions
# ============================================================================

# A) Match rate global
match_counts = (
    df_mkt["match_label"].value_counts(dropna=False).rename_axis("match_label").reset_index(name="count")
)
match_counts["pct"] = (match_counts["count"] / match_counts["count"].sum() * 100).round(1)

fig_match = px.pie(
    match_counts,
    names="match_label",
    values="count",
    hole=0.45,
    color="match_label",
    color_discrete_map={"Match": "#1D9E75", "Pas de match": "#E24B4A"},
    title="Match rate global"
)
fig_match.update_traces(textinfo="percent+label")
fig_match.show()
export_figure(fig_match, "viz_match_rate_global")

# B) Importance déclarée vs genre (bar chart)
declared_cols = [c for c in DECLARED_MAP if c in df_mkt.columns]
imp_gender = (
    df_mkt.groupby("gender_label", dropna=False)[declared_cols]
    .mean()
    .reset_index()
    .melt(id_vars="gender_label", var_name="variable", value_name="importance_declaree")
)
imp_gender["variable_label"] = imp_gender["variable"].map(DECLARED_MAP)

fig_decl_gender = px.bar(
    imp_gender,
    x="variable_label",
    y="importance_declaree",
    color="gender_label",
    barmode="group",
    title="Importance declarée vs genre",
    labels={"variable_label": "Critere", "importance_declaree": "Importance moyenne (0-10)", "gender_label": "Genre"},
    color_discrete_map={"Femme": "#D4537E", "Homme": "#378ADD"}
)
fig_decl_gender.show()
export_figure(fig_decl_gender, "viz_importance_declaree_vs_genre")

# C) Distribution des variables clés (boxplots simples)
key_cols = [c for c in ["attr", "fun", "intel", "sinc", "amb", "like"] if c in df_mkt.columns]
dist_df = df_mkt[key_cols].melt(var_name="variable", value_name="score")
dist_df["variable"] = dist_df["variable"].map(lambda x: ATTR_MAP.get(x, x))

fig_dist = px.box(
    dist_df,
    x="variable",
    y="score",
    points=False,
    color="variable",
    title="Distribution des variables clés"
)
fig_dist.update_layout(showlegend=False)
fig_dist.show()
export_figure(fig_dist, "viz_distribution_variables_cles")

Export image (kaleido): ../outputs/images/viz_match_rate_global.png


Export image (kaleido): ../outputs/images/viz_importance_declaree_vs_genre.png


Export image (kaleido): ../outputs/images/viz_distribution_variables_cles.png


### Observations univariées (lecture marketing)

- **Le taux de match global est structurellement minoritaire** : la plupart des interactions ne convertissent pas en match.
- **Les priorités déclarées diffèrent peu à peu selon le genre**, mais certaines dimensions (souvent attractivité/fun) ressortent davantage.
- **Les distributions de scores montrent une forte variabilité individuelle** : cela indique qu’une stratégie unique “one-size-fits-all” serait sous-optimale.
- **Implication marketing** : il faut penser en segments (messages et offres) plutôt qu’en message unique.


## 2) Analyses bivariées

Ces graphes croisent les variables avec le `match` pour identifier des relations utiles à l’action.

In [33]:
# ============================================================================
# BIVARIÉ : Corrélations, Match vs Attractivité, Gaps partenaires, Déclaratif vs Réel
# ============================================================================

# A) Corrélation avec le match (heatmap)
corr_cols = [c for c in ["match", "attr", "fun", "intel", "sinc", "amb", "shar", "like", "prob", "attr_o", "like_o", "prob_o", "int_corr"] if c in df_mkt.columns]
corr_matrix = df_mkt[corr_cols].corr(numeric_only=True)

fig_corr = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Corrélation avec le match (et entre variables clés)"
)
fig_corr.show()
export_figure(fig_corr, "viz_correlation_heatmap_match")

# B) Match vs Attractivite (bar par niveau)
attr_bins = [0, 2, 4, 6, 8, 10]
attr_labels = ["0-2", "2-4", "4-6", "6-8", "8-10"]
if "attr" in df_mkt.columns:
    tmp_attr = df_mkt.dropna(subset=["attr", "match"]).copy()
    tmp_attr["attr_bin"] = pd.cut(tmp_attr["attr"], bins=attr_bins, labels=attr_labels, include_lowest=True)
    match_attr = tmp_attr.groupby("attr_bin", observed=False)["match"].mean().reset_index()
    match_attr["match_rate_pct"] = (match_attr["match"] * 100).round(1)

    fig_match_attr = px.bar(
        match_attr,
        x="attr_bin",
        y="match_rate_pct",
        title="Match vs Attractivite",
        labels={"attr_bin": "Niveau d'attractivite", "match_rate_pct": "Taux de match (%)"},
        color="match_rate_pct",
        color_continuous_scale="Teal"
    )
    fig_match_attr.show()
    export_figure(fig_match_attr, "viz_match_vs_attractivite")

# C) Différences entre partenaires (attractivité + intérêt)
gap_cols = [c for c in ["gap_attr", "gap_like", "match"] if c in df_mkt.columns]
if set(["gap_attr", "gap_like", "match"]).issubset(gap_cols):
    gap_summary = (
        df_mkt.groupby("match")[ ["gap_attr", "gap_like"] ]
        .mean()
        .reset_index()
        .melt(id_vars="match", var_name="gap_type", value_name="gap_moyen")
    )
    gap_summary["gap_type"] = gap_summary["gap_type"].map({"gap_attr": "Différence d'attractivite", "gap_like": "Différence d'interet"})
    gap_summary["match_label"] = gap_summary["match"].map({0: "Pas de match", 1: "Match"})

    fig_gap = px.bar(
        gap_summary,
        x="gap_type",
        y="gap_moyen",
        color="match_label",
        barmode="group",
        title="Différences entre partenaires",
        labels={"gap_moyen": "Ecart moyen", "gap_type": "Indicateur"},
        color_discrete_map={"Match": "#1D9E75", "Pas de match": "#E24B4A"}
    )
    fig_gap.show()
    export_figure(fig_gap, "viz_differences_entre_partenaires")

# D) Déclaratif vs réel (bar chart comparatif)
pairs_decl_real = [
    ("attr1_1", "attr", "Attractivite"),
    ("sinc1_1", "sinc", "Sincerite"),
    ("intel1_1", "intel", "Intelligence"),
    ("fun1_1", "fun", "Fun"),
    ("amb1_1", "amb", "Ambition"),
    ("shar1_1", "shar", "Interets communs"),
]
rows_cmp = []
for d_col, r_col, label in pairs_decl_real:
    if d_col in df_mkt.columns and r_col in df_mkt.columns:
        rows_cmp.append({
            "Variable": label,
            "Importance declaree": df_mkt[d_col].mean(),
            "Importance reelle": df_mkt[r_col].mean(),
        })

cmp_df = pd.DataFrame(rows_cmp)
cmp_long = cmp_df.melt(id_vars="Variable", var_name="Type", value_name="Score")
fig_decl_real = px.bar(
    cmp_long,
    x="Variable",
    y="Score",
    color="Type",
    barmode="group",
    title="Declaratif vs Reel",
    color_discrete_map={"Importance declaree": "#7F77DD", "Importance reelle": "#EF9F27"}
)
fig_decl_real.show()
export_figure(fig_decl_real, "viz_declaratif_vs_reel")

# E) Analyse par genre (homme->femme / femme->homme)
seg_df = (
    df_mkt.groupby("segment_genre", dropna=False)["match"]
    .mean()
    .reset_index()
)
seg_df["match_rate_pct"] = (seg_df["match"] * 100).round(1)
fig_seg = px.bar(
    seg_df,
    x="segment_genre",
    y="match_rate_pct",
    color="segment_genre",
    title="Analyse par genre",
    labels={"segment_genre": "Segment", "match_rate_pct": "Taux de match (%)"},
    color_discrete_map={"Homme -> Femme": "#378ADD", "Femme -> Homme": "#D4537E"}
)
fig_seg.update_layout(showlegend=False)
fig_seg.show()
export_figure(fig_seg, "viz_analyse_par_genre")

Export image (kaleido): ../outputs/images/viz_correlation_heatmap_match.png


Export image (kaleido): ../outputs/images/viz_match_vs_attractivite.png


Export image (kaleido): ../outputs/images/viz_differences_entre_partenaires.png


Export image (kaleido): ../outputs/images/viz_declaratif_vs_reel.png


Export image (kaleido): ../outputs/images/viz_analyse_par_genre.png


### Observations bivariées (interprétation)

- **Match vs attractivité** : le taux de match augmente généralement avec les niveaux d’attractivité perçue.
- **Corrélations** : `like` / `prob` sont souvent plus proches du match que les préférences déclarées initiales.
- **Différences entre partenaires** : plus l’écart de perception est faible, plus la conversion en match tend à être favorable.
- **Déclaratif vs réel** : les préférences exprimées en amont ne se traduisent pas parfaitement dans les évaluations réelles en interaction.
- **Segmentation genre** : les patterns sont proches mais pas identiques entre `Homme -> Femme` et `Femme -> Homme`, ce qui justifie des messages marketing différenciés.


In [34]:
# ============================================================================
# DASHBOARD FINAL DE SYNTHÈSE (simple et lisible)
# ============================================================================

# KPI et données de synthèse
match_rate_global = float(df_mkt["match"].mean() * 100)

# Reprend des structures déjà calculées
corr_with_match = (
    df_mkt[[c for c in ["match", "attr", "fun", "intel", "sinc", "amb", "shar", "like", "prob"] if c in df_mkt.columns]]
    .corr(numeric_only=True)["match"]
    .drop("match")
    .sort_values(ascending=False)
)

best_attr = corr_with_match.index[0] if len(corr_with_match) > 0 else "n/a"
best_corr = corr_with_match.iloc[0] if len(corr_with_match) > 0 else 0.0

# Dashboard 2x3
fig_dash_mkt = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}]],
    subplot_titles=(
        "Match rate global",
        "Importance déclarée vs genre",
        "Match vs Attractivité",
        "Corrélation avec match",
        "Déclaratif vs Réel (moyennes)",
        "Segmentation genre"
    )
)

# 1) Donut match
fig_dash_mkt.add_trace(
    go.Pie(
        labels=match_counts["match_label"],
        values=match_counts["count"],
        hole=0.55,
        marker=dict(colors=["#1D9E75", "#E24B4A"]),
        textinfo="percent"
    ),
    row=1, col=1
)

# 2) Importance déclarée: moyenne globale par critère
imp_global = (
    df_mkt[[c for c in DECLARED_MAP if c in df_mkt.columns]]
    .mean()
    .rename(index=DECLARED_MAP)
    .sort_values(ascending=False)
)
fig_dash_mkt.add_trace(
    go.Bar(x=imp_global.index.tolist(), y=imp_global.values.tolist(), marker_color="#7F77DD", showlegend=False),
    row=1, col=2
)

# 3) Match vs attractivité
if "attr_bin" in locals() or "match_attr" in locals():
    fig_dash_mkt.add_trace(
        go.Bar(x=match_attr["attr_bin"].astype(str), y=match_attr["match_rate_pct"], marker_color="#1D9E75", showlegend=False),
        row=1, col=3
    )

# 4) Corr avec match
fig_dash_mkt.add_trace(
    go.Bar(x=corr_with_match.values[::-1], y=[ATTR_MAP.get(c, c) for c in corr_with_match.index[::-1]], orientation="h", marker_color="#378ADD", showlegend=False),
    row=2, col=1
)

# 5) Declaratif vs réel (delta)
cmp_df["Delta reel - declare"] = cmp_df["Importance reelle"] - cmp_df["Importance declaree"]
fig_dash_mkt.add_trace(
    go.Bar(x=cmp_df["Variable"], y=cmp_df["Delta reel - declare"], marker_color="#EF9F27", showlegend=False),
    row=2, col=2
)

# 6) Segmentation genre
fig_dash_mkt.add_trace(
    go.Bar(x=seg_df["segment_genre"], y=seg_df["match_rate_pct"], marker_color=["#378ADD", "#D4537E"], showlegend=False),
    row=2, col=3
)

fig_dash_mkt.update_layout(
    height=850,
    title=f"Dashboard marketing — lecture simplifiée (match global: {match_rate_global:.1f}% | meilleur signal: {ATTR_MAP.get(best_attr, best_attr)} r={best_corr:.2f})",
    plot_bgcolor="white"
)
fig_dash_mkt.update_yaxes(title_text="Importance moyenne", row=1, col=2)
fig_dash_mkt.update_yaxes(title_text="Taux de match (%)", row=1, col=3)
fig_dash_mkt.update_yaxes(title_text="Variables", row=2, col=1)
fig_dash_mkt.update_xaxes(title_text="Corrélation", row=2, col=1)
fig_dash_mkt.update_yaxes(title_text="Delta", row=2, col=2)
fig_dash_mkt.update_yaxes(title_text="Taux de match (%)", row=2, col=3)

fig_dash_mkt.show()
export_figure(fig_dash_mkt, "dashboard_marketing_synthese")

Export image (kaleido): ../outputs/images/dashboard_marketing_synthese.png


### Insights actionnables pour marketing

- **Positionnement produit** : valoriser les dimensions qui expliquent le mieux le match (souvent attractivité, fun, intérêt relationnel).
- **Messaging** : adapter les promesses de communication aux comportements réels (pas uniquement au déclaratif).
- **Segmentation CRM** : activer des variantes de messages selon `Homme -> Femme` et `Femme -> Homme`.
- **Optimisation funnel** : utiliser des signaux simples (niveau de `like`, cohérence des perceptions entre partenaires) pour prioriser les relances.
- **Pilotage** : suivre ce dashboard chaque itération pour mesurer l’évolution des leviers de conversion.


## Couverture explicite des questions du projet

Cette section répond point par point aux `Data Exploration Ideas` du brief, avec visualisations simples et interprétations business.

In [ ]:
# ============================================================================
# RÉPONSES DIRECTES AUX 5 QUESTIONS DU BRIEF
# ============================================================================

# Mapping utilitaire
attr_pairs = [
    ("attr1_1", "attr", "Attractivité"),
    ("sinc1_1", "sinc", "Sincérité"),
    ("intel1_1", "intel", "Intelligence"),
    ("fun1_1", "fun", "Fun"),
    ("amb1_1", "amb", "Ambition"),
    ("shar1_1", "shar", "Intérêts communs"),
]

# 1) Least desirable attributes, par genre (déclaratif)
pref_cols = [p for p, _, _ in attr_pairs if p in df_mkt.columns]
pref_gender = (
    df_mkt.groupby("gender_label", dropna=False)[pref_cols]
    .mean()
    .T
    .reset_index()
    .rename(columns={"index": "pref_col"})
)
pref_gender["Attribut"] = pref_gender["pref_col"].map(dict((p, l) for p, _, l in attr_pairs))

pref_long = pref_gender.melt(id_vars=["pref_col", "Attribut"], var_name="Genre", value_name="Importance déclarée")
fig_q1 = px.bar(
    pref_long,
    x="Attribut",
    y="Importance déclarée",
    color="Genre",
    barmode="group",
    title="Q1 — Attributs les moins désirés (déclaratif) selon le genre",
    color_discrete_map={"Femme": "#D4537E", "Homme": "#378ADD"}
)
fig_q1.show()
export_figure(fig_q1, "q1_least_desirable_by_gender")

# 2) Attractivité déclarée vs impact réel
attr_decl_real = pd.DataFrame({
    "Mesure": ["Importance déclarée (attr1_1)", "Impact réel (corr attr vs match)"],
    "Valeur": [
        float(df_mkt["attr1_1"].mean()) if "attr1_1" in df_mkt.columns else np.nan,
        float(df_mkt["attr"].corr(df_mkt["match"])) if {"attr", "match"}.issubset(df_mkt.columns) else np.nan,
    ],
})
fig_q2 = px.bar(
    attr_decl_real,
    x="Mesure", y="Valeur",
    title="Q2 — Attractivité : importance déclarée vs impact réel",
    color="Mesure",
    color_discrete_sequence=["#7F77DD", "#1D9E75"],
)
fig_q2.update_layout(showlegend=False)
fig_q2.show()
export_figure(fig_q2, "q2_attractiveness_declared_vs_real")

# 3) Intérêts communs vs même race
q3 = pd.DataFrame({
    "Variable": ["Intérêts communs (int_corr)", "Même race (samerace)"],
    "Corrélation avec match": [
        float(df_mkt["int_corr"].corr(df_mkt["match"])) if {"int_corr", "match"}.issubset(df_mkt.columns) else np.nan,
        float(df_mkt["samerace"].corr(df_mkt["match"])) if {"samerace", "match"}.issubset(df_mkt.columns) else np.nan,
    ],
})
fig_q3 = px.bar(
    q3,
    x="Variable", y="Corrélation avec match",
    title="Q3 — Intérêts communs vs même race",
    color="Variable",
    color_discrete_sequence=["#378ADD", "#E24B4A"],
)
fig_q3.update_layout(showlegend=False)
fig_q3.show()
export_figure(fig_q3, "q3_shared_interests_vs_race")

# 4) Les participants prédisent-ils leur valeur perçue ? (calibration prob vs match)
if {"prob", "match"}.issubset(df_mkt.columns):
    calib = df_mkt[["prob", "match"]].dropna().copy()
    calib["prob_bin"] = pd.cut(calib["prob"], bins=[0,2,4,6,8,10], include_lowest=True)
    calib_grp = calib.groupby("prob_bin", observed=False).agg(
        match_rate=("match", "mean"),
        prob_moy=("prob", "mean"),
        n=("match", "size"),
    ).reset_index()
    calib_grp["match_rate_pct"] = (calib_grp["match_rate"] * 100).round(1)

    fig_q4 = px.line(
        calib_grp,
        x="prob_moy", y="match_rate_pct",
        markers=True,
        title="Q4 — Calibration : probabilité perçue vs taux de match réel",
        labels={"prob_moy": "Probabilité perçue moyenne", "match_rate_pct": "Taux de match réel (%)"},
    )
    fig_q4.add_trace(go.Scatter(
        x=[0, 10], y=[0, 100],
        mode="lines", name="Calibration parfaite",
        line=dict(color="#888780", dash="dash")
    ))
    fig_q4.show()
    export_figure(fig_q4, "q4_perceived_value_calibration")

# 5) First date vs last date
order_cols = [c for c in df_mkt.columns if any(k in c.lower() for k in ["order", "position", "first", "last", "date_no", "round"])]
print("Q5 — Variables d'ordre disponibles:", order_cols)
if not order_cols:
    print("Q5 — Non testable avec le dataset courant: aucune variable d'ordre first/last n'est disponible.")

### Interprétation (lien avec la décision de second date)

- **Q1 — Least desirable attributes** : les attributs en bas de classement (déclaratif) sont visibles par genre et permettent de cibler les messages les moins différenciants.
- **Q2 — Attractivité déclarée vs impact réel** : l’attractivité est importante dans le discours, mais son effet réel doit être lu via la relation au `match`.
- **Q3 — Intérêts communs vs race** : la comparaison des corrélations montre quel facteur est réellement plus informatif pour le second date.
- **Q4 — Valeur perçue / calibration** : la courbe `prob perçue` vs `match réel` indique si les participants se surestiment ou se sous-estiment.
- **Q5 — First vs last** : non mesurable ici faute de variable d’ordre dans les données disponibles (point à signaler dans la conclusion du projet).

**Deliverable checklist**
- Descriptive statistics : présentes dans les sections précédentes + cette section.
- Visualisations : présentes et exportées en PNG dans `outputs/images`.
- Captions / interprétations : ajoutées dans les cellules Markdown, orientées `pourquoi second date`.
